# 实验二：房源采集与 MySQL 存储

《大数据采集与处理》课程实验二。

采集对象：安居客教学站 `http://www.bspider.top/anjuke/`。

流程：网页采集（requests + BeautifulSoup + CSS 选择器）→ 下一页翻页 → 保存 CSV（csv.writer, utf-8-sig）→ 读取 CSV 写入 MySQL（PyMySQL）。

字段顺序（与建表一致）：

`title → house → address → struct → area → floor → maketime → price`

> MySQL 入库部分的单元格已标注 `mysql` 标签，需在**课程 JupyterLab**中运行（数据库服务名 `mysql` 只在课程 Docker 网络内可解析）。本机直接运行会连接失败，这是正常的。

In [1]:
import csv
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "http://www.bspider.top/anjuke/"
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

# 反爬实测：极简 UA（手册示例的 "Mozilla/5.0"）会命中安居客的访问验证页，
# 必须使用完整的浏览器请求头才能拿到真实列表页。
HEADERS_MINIMAL = {"User-Agent": "Mozilla/5.0"}
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9",
}
print("依赖导入完成，字段顺序:", FIELDS)

依赖导入完成，字段顺序: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']


## 一、获取第一页并确认内容

按手册先用极简 UA 请求一次并检查——**即使状态码 200，也可能是访问验证页**。

In [ ]:
def fetch(url, headers, tries=3, timeout=60):
    """带重试的 GET：教学站响应较慢（首次可达 20 秒），一次超时不代表不可访问。"""
    last = None
    for i in range(1, tries + 1):
        try:
            return requests.get(url, headers=headers, timeout=timeout)
        except requests.RequestException as e:
            last = e
            print("  第 %d 次请求失败：%s，2 秒后重试…" % (i, type(e).__name__))
            time.sleep(2)
    raise last


# 1) 极简 UA：预期被反爬拦成"访问验证"页（即使状态码是 200）
try:
    r_min = fetch(BASE_URL, HEADERS_MINIMAL)
    print("极简 UA -> status:", r_min.status_code, "| 长度:", len(r_min.text),
          "| 是否验证页:", "访问验证" in r_min.text)
except requests.RequestException as e:
    print("极简 UA 对照请求失败（仅影响对照演示，不影响后续采集）:", type(e).__name__)

# 2) 完整浏览器请求头：拿到真实房源列表
r = fetch(BASE_URL, HEADERS)
r.raise_for_status()
r.encoding = r.apparent_encoding
html = r.text
soup = BeautifulSoup(html, "html.parser")

print("完整 UA -> status:", r.status_code, "| 长度:", len(html))
print("网页标题:", soup.title.get_text(strip=True))
print("是否验证页:", "访问验证" in html)
assert "访问验证" not in html, "命中访问验证页，停止采集"
print("确认拿到真实房源列表页")

## 二、网页分析与房源解析

### 2.1 找到“一条房源”的父容器

F12 → Elements → Copy selector 复制标题节点的原始选择器会得到一长串带 `:nth-child()` 的路径，页面结构稍变即失效。向上追溯到刚好包住整条房源（标题、楼盘、地址、属性、价格都在内）的节点，实测为 **`div.property`**——每条房源一个，class 稳定。用相对选择器在它内部找字段即可。

In [5]:
rows = soup.select("div.property")
print("div.property 数量:", len(rows))

div.property 数量: 60


In [6]:
row = rows[0]
node = row.select_one("h3")
print("第一条标题:", node.get_text(strip=True) if node else None)

第一条标题: 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器


### 2.2 逐字段定位

一条房源内部结构（实测）：

| 字段 | 选择器 | 说明 |
|---|---|---|
| title | `h3.property-content-title-name` | 卖点标题 |
| house | `p.property-content-info-comm-name` | 楼盘名 |
| address | `p.property-content-info-comm-address` | 文字分散在多个 `<span>`，`get_text` 直接拼接，**不主动插空格** |
| struct | `p.property-content-info-attribute` | 数字被拆进多个 span，`get_text` 拼回 "3室2厅1卫" |
| area | 属性组中含 `㎡` 的项 | 保留单位 |
| floor | 属性组中含 `层` 的项 | 朝向（"南北"）排在楼层前面且不含"层"，不能按下标硬取 |
| maketime | 属性组中匹配 `\d{4}年` 的项 | 可能缺失，缺失填空字符串 |
| price | `p.property-price-average` | 单价"元/㎡"；`property-price-total` 是总价"万"，**不取** |

In [8]:
# 打印第一条房源属性组中的每一项，确认顺序
info = None
for div in row.select("div.property-content-info"):
    # 排除楼盘信息块 property-content-info-comm，只留属性组
    if "property-content-info-comm" not in div.get("class", []):
        info = div
        break
items = [p.get_text(strip=True) for p in info.select("p.property-content-info-text")]
print("属性组共 %d 项:" % len(items))
for i, t in enumerate(items):
    print("  [%d] %r" % (i, t))

属性组共 5 项:
  [0] '3室2厅1卫'
  [1] '89.9㎡'
  [2] '南北'
  [3] '低层(共27层)'
  [4] '2019年建造'


In [9]:
def parse_house(row):
    """解析一条房源，按 title→house→address→struct→area→floor→maketime→price 顺序返回 8 项"""
    el = row.select_one("h3.property-content-title-name")
    title = el.get_text(strip=True) if el else ""

    el = row.select_one("p.property-content-info-comm-name")
    house = el.get_text(strip=True) if el else ""

    # 地址分散在多个 <span>，get_text(strip=True) 按原有文字拼接，不插入空格
    el = row.select_one("p.property-content-info-comm-address")
    address = el.get_text(strip=True) if el else ""

    # 属性组：第一个不带 comm 的 property-content-info 块
    info = None
    for div in row.select("div.property-content-info"):
        if "property-content-info-comm" not in div.get("class", []):
            info = div
            break
    texts = [p.get_text(strip=True) for p in info.select("p.property-content-info-text")] if info else []

    # 户型：带 -attribute 类的那个 <p>（数字拆在多个 span 里，get_text 拼回）
    el = info.select_one("p.property-content-info-attribute") if info else None
    struct = el.get_text(strip=True) if el else ""

    # 面积/楼层/年代按内容特征识别，避免把朝向当成楼层
    area, floor, maketime = "", "", ""
    for t in texts:
        if t == struct:
            continue
        if not area and ("㎡" in t or "平米" in t):
            area = t                      # 面积：保留单位
        elif not floor and "层" in t:
            floor = t                     # 楼层：如 低层(共27层)
        elif not maketime and re.match(r"^\d{4}年", t):
            maketime = t                  # 建造年代，如 2019年建造
    # 其余未匹配项是朝向（如"南北"），不属于 8 个字段，丢弃

    # 单价：property-price-average 是"元/㎡"，property-price-total 是总价，不取
    el = row.select_one("p.property-price-average")
    price = el.get_text(strip=True) if el else ""

    return [title, house, address, struct, area, floor, maketime, price]

record = parse_house(rows[0])
for k, v in zip(FIELDS, record):
    print("%-9s: %s" % (k, v))

title    : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house    : 万科金域学府翰林菁华里
address  : 南岸学府大道二塘路59号
struct   : 3室2厅1卫
area     : 89.9㎡
floor    : 低层(共27层)
maketime : 2019年建造
price    : 16465元/㎡


In [10]:
# 首 条、第 2 条、最后一条逐字段对照
for idx in [0, 1, len(rows) - 1]:
    print("---- 第 %d 条 ----" % (idx + 1))
    for k, v in zip(FIELDS, parse_house(rows[idx])):
        print("  %-9s: %s" % (k, v))

---- 第 1 条 ----
  title    : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house    : 万科金域学府翰林菁华里
  address  : 南岸学府大道二塘路59号
  struct   : 3室2厅1卫
  area     : 89.9㎡
  floor    : 低层(共27层)
  maketime : 2019年建造
  price    : 16465元/㎡
---- 第 2 条 ----
  title    : 黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住
  house    : 两代一家
  address  : 渝北渝北黄泥磅紫荆路29号
  struct   : 3室2厅2卫
  area     : 94.5㎡
  floor    : 低层(共28层)
  maketime : 2008年建造
  price    : 14815元/㎡
---- 第 60 条 ----
  title    : 精装修 电梯房 4室2厅 91平
  house    : 华润凯旋天地
  address  : 南岸弹子石腾龙大道45号
  struct   : 4室2厅2卫
  area     : 91.3㎡
  floor    : 低层(共32层)
  maketime : 2015年建造
  price    : 15886元/㎡


### 2.3 完整性检查

手册要求：面积或单价缺失时先查原因。对整页 60 条做一次全字段非空检查（maketime 允许为空）。

In [12]:
missing = []
for i, row in enumerate(rows):
    rec = parse_house(row)
    for k, v in zip(FIELDS, rec):
        if v == "" and k != "maketime":
            missing.append((i, k))
print("非年代字段缺失数:", len(missing))
if missing:
    print(missing)
mt_missing = sum(1 for row in rows if parse_house(row)[6] == "")
print("maketime 缺失条数:", mt_missing, "/ 60（允许，填空字符串）")

非年代字段缺失数: 0
maketime 缺失条数: 0 / 60（允许，填空字符串）


## 三、采集下一页

下一页按钮：`a.next.next-active`（class 同时含 `next` 与 `next-active` 表示可点）。
`get_text()` 取到的是"下一页"三个字，地址要用 `.get("href")` 取；href 只有 `?page=2` 这样的相对值，用 `urljoin` 补全。

In [14]:
nxt = soup.select_one("a.next")
href = nxt.get("href")
print("下一页文字:", nxt.get_text(strip=True))
print("下一页 href 原值:", href)
print("class:", nxt.get("class"))
next_url = urljoin(r.url, href)
print("urljoin 补全后:", next_url)

下一页文字: 下一页
下一页 href 原值: ?page=2
class: ['next', 'next-active']
urljoin 补全后: http://www.bspider.top/anjuke/?page=2


In [ ]:
# 分页循环：请求 → 解析 → 找下一页 → 补全 → 去重/越界检查 → 限速
session = requests.Session()
session.headers.update(HEADERS)

visited = set()
all_records = []
page_logs = []
MAX_PAGES = 30          # 安全页数上限，防无限循环
stop_reason = None
current_url = BASE_URL
page_no = 0

while current_url and page_no < MAX_PAGES:
    page_no += 1
    for _try in range(1, 4):                      # 网络抖动时重试，最多 3 次
        try:
            resp = session.get(current_url, timeout=60)
            break
        except requests.RequestException as e:
            print("第 %d 页第 %d 次请求失败：%s" % (page_no, _try, type(e).__name__))
            if _try == 3:
                raise
            time.sleep(3)
    resp.encoding = resp.apparent_encoding
    if "访问验证" in resp.text:
        stop_reason = "命中访问验证页，异常中止"
        page_logs.append((page_no, current_url, 0, len(all_records), stop_reason))
        break
    sp = BeautifulSoup(resp.text, "html.parser")
    page_rows = sp.select("div.property")
    if not page_rows:
        stop_reason = "页面无房源（结构异常），中止"
        page_logs.append((page_no, current_url, 0, len(all_records), stop_reason))
        break
    for row_ in page_rows:
        all_records.append(parse_house(row_))
    visited.add(current_url)
    note = ""
    nxt = sp.select_one("a.next")
    if nxt is None or not nxt.get("href"):
        stop_reason = "下一页按钮不存在（分页区结束）"
        page_logs.append((page_no, current_url, len(page_rows), len(all_records), note or stop_reason))
        break
    if "next-active" not in (nxt.get("class") or []):
        stop_reason = "下一页按钮已禁用（已到末页），采集完成"
        page_logs.append((page_no, current_url, len(page_rows), len(all_records), stop_reason))
        break
    cand = urljoin(resp.url, nxt.get("href"))
    if "bspider.top" not in cand:
        stop_reason = "下一页地址越出教学站，中止"
        page_logs.append((page_no, current_url, len(page_rows), len(all_records), stop_reason))
        break
    if cand in visited:
        stop_reason = "下一页地址重复，中止"
        page_logs.append((page_no, current_url, len(page_rows), len(all_records), stop_reason))
        break
    page_logs.append((page_no, current_url, len(page_rows), len(all_records), "-> " + cand))
    current_url = cand
    time.sleep(1.2)      # 限速：每次请求间隔至少 1 秒

if stop_reason is None:
    stop_reason = "达到安全页数上限 %d，中止" % MAX_PAGES

print("%-4s %-46s %6s %8s  %s" % ("页码", "URL", "本页", "累计", "说明"))
for lg in page_logs:
    print("%-5d %-46s %6d %8d  %s" % lg)
print()
print("停止原因:", stop_reason)
print("总页数:", page_no, "| 采集记录总数（去重前）:", len(all_records))

## 四、保存 CSV 并核对

要求：`csv.writer`、编码 `utf-8-sig`、`newline=""`、8 字段表头、按 8 字段完全相同去重并记录前后条数。

In [17]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / "houseinfo.csv"

before_dedup = len(all_records)
seen = set()
unique_records = []
for rec in all_records:
    key = tuple(rec)
    if key not in seen:
        seen.add(key)
        unique_records.append(rec)
after_dedup = len(unique_records)
print("去重前:", before_dedup, "条 | 完全相同的重复:", before_dedup - after_dedup, "条 | 去重后:", after_dedup, "条")

with open(CSV_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)              # 表头：8 个字段名，顺序一致
    for rec in unique_records:
        assert len(rec) == 8, "每条记录必须恰好 8 项"
        writer.writerow(rec)
print("已保存:", CSV_PATH, "(%.1f KB)" % (CSV_PATH.stat().st_size / 1024))

去重前: 600 条 | 完全相同的重复: 102 条 | 去重后: 498 条
已保存: data\houseinfo.csv (92.2 KB)


In [18]:
# 用 DictReader 回读核对：表头不计入数据行数
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("回读表头:", reader.fieldnames)
    assert reader.fieldnames == FIELDS, "表头与 8 字段顺序不一致"
    rows_back = list(reader)
print("回读数据行数（不含表头）:", len(rows_back))

print()
print("首条:", {k: rows_back[0][k] for k in FIELDS})
print("跨页一条(第65条):", {k: rows_back[64][k] for k in FIELDS})
print("末条:", {k: rows_back[-1][k] for k in FIELDS})

回读表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
回读数据行数（不含表头）: 498

首条: {'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}
跨页一条(第65条): {'title': '万科新出  三房 153万买家具家电齐全 核心位置', 'house': '万科金色悦城(1期)', 'address': '沙坪坝凤天路凤鸣山300号', 'struct': '3室2厅1卫', 'area': '94.6㎡', 'floor': '低层(共33层)', 'maketime': '2016年建造', 'price': '16182元/㎡'}
末条: {'title': '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2015年建造', 'price': '15530元/㎡'}


## 五、MySQL 数据存储（在课程 JupyterLab 中运行）

以下单元格带 `mysql` 标签。连接参数为课程教学环境：`host=mysql, port=3306, user=student, password=123456, database=smartpark_lab`。**请先在 MySQL 终端完成建表**（手册第一阶段），再运行本节。

流程：读 CSV → 连接测试 → 查库内已有记录防重复插入 → `executemany` 批量写入 → `commit`；出错 `rollback`；最后关闭游标和连接。

In [ ]:
import pymysql

CSV_PATH = Path("data/houseinfo.csv")

# 1. 读 CSV，检查表头，按 8 字段固定顺序整理记录
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == FIELDS, "CSV 表头与 8 字段顺序不一致"
    csv_records = [tuple(row[k] for k in FIELDS) for row in reader]
print("从 CSV 读入记录:", len(csv_records), "条")

# 2. 连接数据库（host=mysql 是课程 Docker 网络内的服务名）
conn = pymysql.connect(
    host="mysql", port=3306,
    user="student", password="123456",
    database="smartpark_lab",
    charset="utf8mb4",
    autocommit=False,          # 由程序显式提交事务
)
print("连接成功:", conn.open)

In [ ]:
# 3. 连接测试：查一条数据确认可用，并记录入库前条数
try:
    with conn.cursor() as cur:
        cur.execute("SELECT VERSION(), DATABASE(), CURRENT_USER()")
        print("连接检查:", cur.fetchone())
        cur.execute("SELECT COUNT(*) FROM houseinfo")
        before_count = cur.fetchone()[0]
        print("入库前条数 before_count:", before_count)

        # 4. 防重复插入：取库内已有的 8 字段组合，只插入不存在的新记录
        cur.execute(
            "SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo"
        )
        existing = set(cur.fetchall())
        new_records = [t for t in csv_records if t not in existing]
        print("库内已有:", len(existing), "条 | CSV 中待插入的新记录:", len(new_records), "条")

        # 5. 批量写入：%s 占位，参数经第二个传入，不做字符串拼接
        sql = ("INSERT INTO houseinfo "
               "(title, house, address, struct, area, floor, maketime, price) "
               "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)")
        if new_records:
            affected = cur.executemany(sql, new_records)
            print("executemany 受影响行数:", affected)
        conn.commit()
        print("已 commit")

        cur.execute("SELECT COUNT(*) FROM houseinfo")
        after_count = cur.fetchone()[0]
        print("入库后条数 after_count:", after_count,
              "| 本次新增:", after_count - before_count)
except Exception as e:
    conn.rollback()          # 出错回滚，撤销未提交的修改
    print("写入失败，已 rollback:", repr(e))
    raise
finally:
    # 6. 成功失败都要释放资源
    conn.close()
    print("连接已关闭")

In [ ]:
# 7. 重复运行验证：再次执行本单元格不应增加任何记录
conn = pymysql.connect(
    host="mysql", port=3306,
    user="student", password="123456",
    database="smartpark_lab", charset="utf8mb4", autocommit=False,
)
try:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM houseinfo")
        c1 = cur.fetchone()[0]
        cur.execute(
            "SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo"
        )
        existing = set(cur.fetchall())
        dup = [t for t in csv_records if t not in existing]
        print("当前库内条数:", c1, "| 本次 Would-insert:", len(dup))
        if dup:
            cur.executemany(
                "INSERT INTO houseinfo "
                "(title, house, address, struct, area, floor, maketime, price) "
                "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)", dup)
        conn.commit()
        cur.execute("SELECT COUNT(*) FROM houseinfo")
        c2 = cur.fetchone()[0]
        print("重跑后条数:", c2, "| 新增:", c2 - c1, "（应为 0）")
        cur.execute(
            "SELECT title, house, address, struct, area, floor, maketime, price "
            "FROM houseinfo ORDER BY id LIMIT 5")
        for row_ in cur.fetchall():
            print(row_)
finally:
    conn.close()
    print("连接已关闭")

## 附：实验信息

- 采集站点：安居客教学站 `http://www.bspider.top/anjuke/`
- 采集方式：requests + BeautifulSoup（html.parser）+ CSS 选择器
- 反爬应对：极简 UA 触发"访问验证"页，改用完整 Chrome 请求头后正常（详见报告"问题与解决"）
- 数据文件：`data/houseinfo.csv`（utf-8-sig，8 字段 + 表头）